In [ ]:
#Here 30 % of labeled dataset is used to train a model then  this train model is used to Good images to train model using active learningimport os, csv, math, numpy as np, tensorflow as tf
from tensorflow import keras
from sklearn.cluster import MiniBatchKMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

# -----------------------------
# 0) Build the VGG16 based model and load weights
# -----------------------------
def vggfun(location):
    # base feature extractor
    base_model = tf.keras.applications.VGG16(
        weights=None, include_top=False, input_shape=(224, 224, 3), pooling='avg')
    base_model.trainable = True

    inputs = keras.Input(shape=(224, 224, 3))
    x = base_model(inputs)  # 512-d embedding

    # bbox prediction head
    x1 = keras.layers.Dense(1024, activation="relu")(x)
    x1 = keras.layers.Dropout(0.5)(x1)
    x1 = keras.layers.Dense(512, activation="relu")(x1)
    x1 = keras.layers.Dropout(0.5)(x1)
    out1 = keras.layers.Dense(1, name="xmin")(x1)
    out2 = keras.layers.Dense(1, name="ymin")(x1)
    out3 = keras.layers.Dense(1, name="xmax")(x1)
    out4 = keras.layers.Dense(1, name="ymax")(x1)

    # class prediction head
    x2 = keras.layers.Dense(1024, activation="relu")(x)
    x2 = keras.layers.Dropout(0.5)(x2)
    x2 = keras.layers.Dense(512, activation="relu")(x2)
    x2 = keras.layers.Dropout(0.5)(x2)
    out_class = keras.layers.Dense(10, activation="softmax", name="class")(x2)

    # final model
    model = keras.models.Model(inputs=inputs, outputs=[out1, out2, out3, out4, out_class])
    model.load_weights(location)
    return model

# -----------------------------
# 1) Build feature extraction model
# -----------------------------
def build_feature_model(full_model: keras.Model) -> keras.Model:
    # Prefer using the GAP layer
    for layer in reversed(full_model.layers):
        if isinstance(layer, tf.keras.layers.GlobalAveragePooling2D):
            return keras.Model(inputs=full_model.input, outputs=layer.output)
    # Otherwise use the class head input
    try:
        class_layer = full_model.get_layer('class')
        feat_tensor = class_layer.input
        return keras.Model(inputs=full_model.input, outputs=feat_tensor)
    except Exception:
        pass
    # Otherwise use bbox head input
    try:
        xmin_layer = full_model.get_layer('xmin')
        feat_tensor = xmin_layer.input
        return keras.Model(inputs=full_model.input, outputs=feat_tensor)
    except Exception:
        pass
    raise RuntimeError("Could not locate embedding tensor")

# -----------------------------
# 2) Dataset loader
# -----------------------------
def make_dataset(root_dir, batch_size=64, shuffle=False):
    # collect image file paths
    exts = (".jpg",".jpeg",".png",".bmp",".tif",".tiff")
    files = [os.path.join(dp,f) for dp,_,fn in os.walk(root_dir) for f in fn if f.lower().endswith(exts)]
    files.sort()
    ds = tf.data.Dataset.from_tensor_slices(files)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(files), reshuffle_each_iteration=False)

    # load and preprocess
    def _load(path):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img,3)
        img = tf.image.resize(img,[224,224])
        img = tf.cast(img,tf.float32) / 255.0
        return img, path

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds, files

# -----------------------------
# 3) Extract embeddings
# -----------------------------
def extract_embeddings_stream(feat_model, ds, N, emb_dim=512):
    # memory map to disk
    embs = np.memmap("embeddings_float32.mmap", mode="w+", dtype=np.float32, shape=(N, emb_dim))
    paths = []
    offset = 0
    for batch, batch_paths in tqdm(ds, desc="Extracting embeddings"):
        feats = feat_model(batch, training=False).numpy()
        bsz = feats.shape[0]
        embs[offset:offset+bsz] = feats
        paths.extend([p.decode("utf-8") for p in batch_paths.numpy()])
        offset += bsz
    return embs, paths

# -----------------------------
# 4) MC-Dropout uncertainty
# -----------------------------
def mc_uncertainty_stream(full_model, ds, N, T=20, eps=1e-9):
    # allocate arrays
    MI = np.zeros((N,), dtype=np.float32)
    H  = np.zeros((N,), dtype=np.float32)
    VR = np.zeros((N,), dtype=np.float32)
    pred_cls = np.zeros((N,), dtype=np.int32)
    pred_std = np.zeros((N,), dtype=np.float32)

    offset = 0
    for imgs, _ in tqdm(ds, desc="MC-Dropout"):
        probs_T = []
        for _ in range(T):
            _, _, _, _, probs = full_model(imgs, training=True)
            probs_T.append(probs.numpy())
        P = np.stack(probs_T, axis=0)
        p_mean = P.mean(axis=0)
        std_all = P.std(axis=0)
        pc = p_mean.argmax(axis=1)
        ps = std_all[np.arange(p_mean.shape[0]), pc]

        H_b = -np.sum(p_mean * np.log(p_mean + eps), axis=1)
        H_exp_b = -np.mean(np.sum(P * np.log(P + eps), axis=2), axis=0)
        MI_b = H_b - H_exp_b
        VR_b = 1.0 - p_mean.max(axis=1)

        bsz = p_mean.shape[0]
        MI[offset:offset+bsz] = MI_b
        H[offset:offset+bsz]  = H_b
        VR[offset:offset+bsz] = VR_b
        pred_cls[offset:offset+bsz] = pc
        pred_std[offset:offset+bsz] = ps
        offset += bsz

    return {"MI": MI, "H": H, "VR": VR, "pred_class": pred_cls, "pred_class_std": pred_std}

# -----------------------------
# 5) Clustering embeddings
# -----------------------------
def cluster_embeddings(embs, method="kmeans", k=64, eps=0.5, min_samples=5):
    X = StandardScaler().fit_transform(embs)
    if method == "kmeans":
        km = MiniBatchKMeans(n_clusters=k, random_state=42, batch_size=4096, n_init="auto")
        labels = km.fit_predict(X)
        return labels, km
    elif method == "dbscan":
        db = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1)
        labels = db.fit_predict(X)
        return labels, db
    else:
        raise ValueError("method must be 'kmeans' or 'dbscan'")

# -----------------------------
# 6) Select top uncertain per cluster
# -----------------------------
def select_per_cluster(labels, scores, top_per_cluster=3, skip_noise_label=-1):
    selected = []
    for cid in np.unique(labels):
        if cid == skip_noise_label:
            continue
        idxs = np.where(labels == cid)[0]
        if idxs.size == 0: continue
        order = idxs[np.argsort(-scores[idxs])]
        selected.extend(order[:top_per_cluster].tolist())
    return selected

# -----------------------------
# 7) Save results to CSV
# -----------------------------
def save_csv(csv_path, paths, labels, unc, top_idxs=None):
    header = ["index","path","cluster","MI","H","VR","pred_class","pred_class_std","selected"]
    with open(csv_path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header)
        for i,p in enumerate(paths):
            sel = 1 if (top_idxs is not None and i in set(top_idxs)) else 0
            w.writerow([i, p, labels[i], float(unc["MI"][i]), float(unc["H"][i]),
                        float(unc["VR"][i]), int(unc["pred_class"][i]), float(unc["pred_class_std"][i]), sel])
    print(f"Saved: {csv_path}")

# -----------------------------
# 8) Main function
# -----------------------------
def main(
    data_root,
    weights_path,
    batch_size=64,
    T=20,
    cluster_method="kmeans",
    k_clusters=64,
    top_per_cluster=3,
    out_csv="al_cluster_scores.csv"
):
    # load model and feature model
    model = vggfun(weights_path)
    feat_model = build_feature_model(model)

    # load dataset
    ds, files = make_dataset(data_root, batch_size=batch_size, shuffle=False)
    N = len(files)
    print(f"Found {N} images.")

    # prepare datasets for embeddings and MC-dropout
    ds_emb, _ = make_dataset(data_root, batch_size=batch_size, shuffle=False)
    ds_mc, _  = make_dataset(data_root, batch_size=batch_size, shuffle=False)

    # extract embeddings
    embs, paths = extract_embeddings_stream(feat_model, ds_emb, N, emb_dim=512)

    # cluster embeddings
    labels, _ = cluster_embeddings(embs, method=cluster_method, k=k_clusters)

    # compute uncertainty
    unc = mc_uncertainty_stream(model, ds_mc, N, T=T)

    # select diverse + uncertain samples
    selected = select_per_cluster(labels, unc["MI"], top_per_cluster=top_per_cluster)

    # save results
    save_csv(out_csv, paths, labels, unc, top_idxs=selected)

    print(f"Selected {len(selected)} images (≈ {top_per_cluster} per cluster).")
    print("Tip: sort the CSV by MI desc within each cluster for manual review / labeling.")

if __name__ == "__main__":
    # parameters
    DATA_ROOT   = r"C:\\Users\\sagar\\remaining"
    WEIGHTS_H5  = r"C:\\Users\\sagar\\final.weights.h5"
    BATCH_SIZE  = 64
    T_SAMPLES   = 20
    METHOD      = "kmeans"
    K_CLUSTERS  = 45
    TOP_PER_C   = 200
    OUT_CSV     = "al_cluster_scores.csv"

    # run main
    main(DATA_ROOT, WEIGHTS_H5, batch_size=BATCH_SIZE, T=T_SAMPLES,
         cluster_method=METHOD, k_clusters=K_CLUSTERS,
         top_per_cluster=TOP_PER_C, out_csv=OUT_CSV)
